# CIFAR-10 ResNet Challenge (5M parameter budget)

Pipeline: cached data loading, pre-activation Wide ResNet under 5M params, SGD + cosine schedule,
TrivialAugment + random erasing + CutMix/MixUp, label smoothing, weight EMA, optional SAM,
optional knowledge distillation from a from-scratch teacher, flip/shift TTA, and `submission.csv` writer.

**How to use**
1. Edit the `CFG` cell. `MODE` is one of `student`, `teacher`, `kd`.
2. Run all cells. Checkpoints, logs, test probabilities and `submission.csv` go to `OUT_DIR`.
3. Typical workflow:
   * Quick experiments: `MODE='student'`, `EPOCHS=40`, `FULL_TRAIN=False` (5k held out validation).
   * Teacher: `MODE='teacher'` (WRN-28-10, about 36M params, allowed only as a teacher).
   * Final: `MODE='kd'`, `FULL_TRAIN=True`, `EPOCHS=200+`, `TEACHER_CKPT` pointing at a teacher that was also trained with `FULL_TRAIN=True`.

On Kaggle: add the competition data, set Accelerator to GPU (T4 or P100), and turn Internet off is fine.
Kaggle sessions stop after 12h, so training saves `last.pt` every epoch and `RESUME=True` continues from it.

In [ ]:
!kaggle competitions files -c cifar-10-res-net-architecture-challenge-dl 2>&1 | head -8; pip install -q -U kagglehub; python -c "import kagglehub; print(kagglehub.competition_download('cifar-10-res-net-architecture-challenge-dl'))" 2>&1 | tail -4

Next Page Token = CfDJ8FwkkMkekkBCtzoFDl4Vj7iKjP6-R_I2xn9vy0n1QmJBA5htDawmElbveU-c_drIiSrZuzGkCqCmShMdTULhPYBfqdknhOT2RtdRq7yK1RRWcUON28F6uWkIRbsEip1c6NB0iOiIKUyRMGz3vd8d43_-8Vvb
name                             size  creationDate                
-------------------------  ----------  --------------------------  
sample_submission.csv           12899  2026-10-26 04:09:59.117000  
test.csv                        36902  2026-10-26 04:09:59.112000  
test_images/test_0000.png        2038  2026-10-26 04:09:59.423000  
test_images/test_0001.png        2042  2026-10-26 04:09:59.422000  
test_images/test_0002.png        2230  2026-10-26 04:09:59.422000  
kagglehub.exceptions.KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl
Please make sure you specified the correct resource identifiers.


## Colab setup (run first)
Before running:
1. **Runtime > Change runtime type > A100 GPU** (Colab Pro).
2. Kaggle token: on kaggle.com go to **Settings > API Tokens > Generate New Token** and copy it. In Colab open the **key icon (Secrets)** in the left sidebar, **Add new secret**, Name `KAGGLE_API_TOKEN`, Value = the token, and switch **Notebook access** on.
3. Running the next cell asks you to allow Google Drive access. Checkpoints are saved to Drive so a disconnected run can resume (`RESUME=True`).

Colab Pro needs this browser tab to stay open while it trains.

In [ ]:
import os, subprocess, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/cifar10_resnet'
os.makedirs(DRIVE_DIR, exist_ok=True)

DATA_DIR = '/content/data'
if not os.path.exists(os.path.join(DATA_DIR, 'train_labels.csv')):
    try:
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')        # new style token (kaggle >= 1.8)
    except Exception:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')          # legacy kaggle.json credentials
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    subprocess.run(['pip', 'install', '-q', '-U', 'kaggle>=1.8.0'], check=True)
    subprocess.run(['kaggle', 'competitions', 'download', '-c', 'cifar-10-res-net-architecture-challenge-dl',
                    '-p', DATA_DIR], check=True)
    for z in [f for f in os.listdir(DATA_DIR) if f.endswith('.zip')]:
        zipfile.ZipFile(os.path.join(DATA_DIR, z)).extractall(DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive


CalledProcessError: Command '['kaggle', 'competitions', 'download', '-c', 'cifar-10-res-net-architecture-challenge-dl', '-p', '/content/data']' returned non-zero exit status 1.

In [ ]:
!until kaggle kernels status thisispragya/notebook732e59142c 2>&1 | grep -qiE 'complete|error'; do sleep 20; done; kaggle kernels status thisispragya/notebook732e59142c; kaggle kernels output thisispragya/notebook732e59142c -p /content/data && ls -la /content/data && cp /content/data/data_cache.npz /content/drive/MyDrive/cifar10_resnet/data_cache.npz && echo CACHE_COPIED

thisispragya/notebook732e59142c has status "KernelWorkerStatus.COMPLETE"
100% 153M/153M [00:19<00:00, 8.10MB/s]

Output file downloaded to /content/data/data_cache.npz
100% 12.6k/12.6k [00:00<00:00, 1.08MB/s]

Output file downloaded to /content/data/sample_submission.csv
100% 36.0k/36.0k [00:00<00:00, 251kB/s]

Output file downloaded to /content/data/test.csv
100% 1.18M/1.18M [00:01<00:00, 866kB/s]

Output file downloaded to /content/data/train_labels.csv
Kernel log downloaded to /content/data/notebook732e59142c.log 
total 157672
drwxr-xr-x 2 root root      4096 Oct 30 19:56 .
drwxr-xr-x 1 root root      4096 Oct 30 19:55 ..
-rw-r--r-- 1 root root 160144726 Oct 30 19:55 data_cache.npz
-rw-r--r-- 1 root root      2276 Oct 30 19:56 notebook732e59142c.log
-rw-r--r-- 1 root root     12899 Oct 30 19:54 sample_submission.csv
-rw-r--r-- 1 root root     36902 Oct 30 19:54 test.csv
-rw-r--r-- 1 root root   1238909 Oct 30 19:55 train_labels.csv
CACHE_COPIED


In [ ]:
import os
# ============================ CONFIG ============================
CFG = dict(
    RUN_NAME      = 'wrn_teacher',
    MODE          = 'teacher',      # 'student' | 'teacher' | 'kd'
    SEED          = 42,

    # data
    DATA_ROOT     = '/content/data',           # None = auto search (/kaggle/input, ./data, ...)
    OUT_DIR       = '/content/drive/MyDrive/cifar10_resnet',
    VAL_SIZE      = 5000,           # stratified hold out from the 50k training images
    FULL_TRAIN    = False,          # True = train on all 50k (no validation), use for final runs
    NUM_WORKERS   = min(8, os.cpu_count()),

    # student architecture (pre-activation WRN). depth = 6n+4, widths per stage
    S_DEPTH       = 22,
    S_WIDTHS      = (72, 144, 264),  # 4,806,954 params. Others: (64,128,256)=4.30M (WRN-22-4), (64,128,272)=4.70M
                                     # depth 16 (80,160,320)=4.29M, depth 28 (48,96,224)=4.16M. Assert below enforces <= 5M
    S_DROPOUT     = 0.0,
    # teacher architecture (no budget, only used to produce soft targets)
    T_DEPTH       = 28,
    T_WIDTHS      = (160, 320, 640),  # WRN-28-10, 36.5M params. Faster teacher: depth 16, (128,256,512) = 11.0M
    T_DROPOUT     = 0.3,

    # optimisation
    EPOCHS        = 200,
    BATCH_SIZE    = 128,
    LR            = 0.1,            # for batch 128, scale linearly with batch size
    MOMENTUM      = 0.9,
    WD            = 5e-4,
    WARMUP_EPOCHS = 5,
    LABEL_SMOOTH  = 0.1,
    SAM           = False,          # +0.5..1% but ~2x slower
    SAM_RHO       = 0.05,

    # augmentation
    TRIVIAL_AUG   = True,
    ERASE_P       = 0.5,            # random erasing (cutout-like)
    MIX_PROB      = 0.5,            # per batch probability of CutMix or MixUp
    CUTMIX_ALPHA  = 1.0,
    MIXUP_ALPHA   = 0.2,
    MIX_OFF_LAST  = 0.1,            # fraction of final epochs with mixing disabled

    # EMA
    EMA_DECAY     = 0.999,

    # knowledge distillation
    TEACHER_CKPT  = '/kaggle/working/wrn_teacher/final_ema.pt',   # from a run with MODE='teacher', RUN_NAME='wrn_teacher'
    KD_ALPHA      = 0.8,
    KD_T          = 4.0,

    # misc
    AMP           = True,
    EVAL_EVERY    = 5,
    TTA_SHIFT     = 2,              # pixel shift for TTA views (0 = flip only). Auto chosen on val when available
    RESUME        = True,
    PARAM_LIMIT   = 5_000_000,
)

In [ ]:
CFG.update(MODE='kd', RUN_NAME='wrn_kd_v1', TEACHER_CKPT='/content/drive/MyDrive/cifar10_resnet/wrn_teacher/final_ema.pt', FULL_TRAIN=False, EPOCHS=200); print(CFG['MODE'], CFG['RUN_NAME'], CFG['TEACHER_CKPT'])

kd wrn_kd_v1 /content/drive/MyDrive/cifar10_resnet/wrn_teacher/final_ema.pt


In [ ]:
!echo token_set=${KAGGLE_API_TOKEN:+yes}; kaggle config view 2>&1 | head; kaggle competitions list --group entered 2>&1 | head -5; kaggle kernels list --mine 2>&1 | head -5

token_set=yes
Configuration values from /root/.config/kaggle
- username: thisispragya
- auth_method: ACCESS_TOKEN
- path: None
- proxy: None
- competition: None
ref                                                                                deadline             category       reward  teamCount  userHasEntered  userRank  
---------------------------------------------------------------------------------  -------------------  ---------  ----------  ---------  --------------  --------  
https://www.kaggle.com/competitions/jigsaw-toxic-comment-classification-challenge  2018-03-20 23:59:00  Featured   35,000 Usd       4539            True         0  
https://www.kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl     2026-11-07 04:59:59  Community       Kudos         22            True         9  
ref                              title               author        lastRunTime                 totalVotes  
-------------------------------  ------------------  ------------  ----

In [ ]:
import os, glob, json, math, time, random, copy
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['SEED'])

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if CFG['OUT_DIR'] == '/kaggle/working' and not os.path.isdir('/kaggle/working'):
    CFG['OUT_DIR'] = './outputs'
RUN_DIR = os.path.join(CFG['OUT_DIR'], CFG['RUN_NAME'])
os.makedirs(RUN_DIR, exist_ok=True)
json.dump(CFG, open(os.path.join(RUN_DIR, 'config.json'), 'w'), indent=2)
torch.backends.cudnn.benchmark = True
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

device: cuda NVIDIA A100-SXM4-40GB


## Data

In [ ]:
def find_data_root():
    if CFG['DATA_ROOT']:
        return CFG['DATA_ROOT']
    for base in ['/kaggle/input', './data', '.', '/content']:
        hits = glob.glob(os.path.join(base, '**', 'train_labels.csv'), recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError('train_labels.csv not found. Set CFG["DATA_ROOT"].')

DATA_ROOT = find_data_root()
print('data root:', DATA_ROOT)

def index_pngs(root):
    idx = {}
    for d, _, files in os.walk(root):
        for f in files:
            if f.endswith('.png'):
                idx[f] = os.path.join(d, f)
    return idx

def load_images(filenames, png_index):
    arr = np.empty((len(filenames), 32, 32, 3), dtype=np.uint8)
    for i, f in enumerate(filenames):
        arr[i] = np.asarray(Image.open(png_index[os.path.basename(f)]).convert('RGB'))
    return arr

CACHE = os.path.join(CFG['OUT_DIR'], 'data_cache.npz')
train_df = pd.read_csv(os.path.join(DATA_ROOT, 'train_labels.csv'))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, 'sample_submission.csv'))

if os.path.exists(CACHE):
    z = np.load(CACHE); X_all, y_all, X_test = z['X'], z['y'], z['Xt']
else:
    t0 = time.time(); pngs = index_pngs(DATA_ROOT)
    X_all = load_images(train_df['filename'].tolist(), pngs)
    y_all = train_df['label'].values.astype(np.int64)
    X_test = load_images(test_df['filename'].tolist(), pngs)
    np.savez(CACHE, X=X_all, y=y_all, Xt=X_test)
    print(f'loaded pngs in {time.time()-t0:.0f}s')
print('train', X_all.shape, 'test', X_test.shape, 'class counts', np.bincount(y_all))

MEAN = X_all.reshape(-1, 3).mean(0, dtype=np.float64) / 255.0   # float64 accumulation (float32 saturates)
STD  = X_all.reshape(-1, 3).std(0, dtype=np.float64) / 255.0
print('mean', MEAN.round(4), 'std', STD.round(4))

# stratified split
rng = np.random.RandomState(CFG['SEED'])
val_idx = np.concatenate([rng.permutation(np.where(y_all == c)[0])[:CFG['VAL_SIZE'] // 10] for c in range(10)])
if CFG['FULL_TRAIN']:
    tr_idx, val_idx = np.arange(len(y_all)), np.array([], dtype=int)
else:
    tr_idx = np.setdiff1d(np.arange(len(y_all)), val_idx)
print('train', len(tr_idx), 'val', len(val_idx))

data root: /content/data
train (50000, 32, 32, 3) test (2000, 32, 32, 3) class counts [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]
mean [0.4914 0.4822 0.4465] std [0.247  0.2435 0.2616]
train 45000 val 5000


In [ ]:
class ArrDS(Dataset):
    def __init__(self, X, y, tf):
        self.X = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous()  # uint8 NCHW
        self.y = None if y is None else torch.from_numpy(y)
        self.tf = tf
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.tf(self.X[i])
        return (x, self.y[i]) if self.y is not None else x

norm = [v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN.tolist(), STD.tolist())]
train_tf = v2.Compose(
    [v2.RandomCrop(32, padding=4, padding_mode='reflect'), v2.RandomHorizontalFlip()]
    + ([v2.TrivialAugmentWide()] if CFG['TRIVIAL_AUG'] else [])
    + norm
    + ([v2.RandomErasing(p=CFG['ERASE_P'], scale=(0.02, 0.25), value=0)] if CFG['ERASE_P'] > 0 else [])
)
test_tf = v2.Compose(norm)

pin = DEVICE == 'cuda'
train_dl = DataLoader(ArrDS(X_all[tr_idx], y_all[tr_idx], train_tf), batch_size=CFG['BATCH_SIZE'], shuffle=True,
                      drop_last=True, num_workers=CFG['NUM_WORKERS'], pin_memory=pin,
                      persistent_workers=CFG['NUM_WORKERS'] > 0)
val_dl = DataLoader(ArrDS(X_all[val_idx], y_all[val_idx], test_tf), batch_size=512, shuffle=False,
                    num_workers=2, pin_memory=pin) if len(val_idx) else None
test_dl = DataLoader(ArrDS(X_test, None, test_tf), batch_size=512, shuffle=False, num_workers=2, pin_memory=pin)

## Model: pre-activation Wide ResNet  (y = S(x) + F(x))

In [ ]:
class PreActBlock(nn.Module):
    def __init__(self, cin, cout, stride, dropout):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin);  self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout); self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        # S(x): identity, or 1x1 projection when shape changes
        self.shortcut = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        s = x if self.shortcut is None else self.shortcut(o)
        o = self.conv1(o)
        o = self.conv2(self.drop(F.relu(self.bn2(o))))
        return o + s

class PreActWRN(nn.Module):
    def __init__(self, depth=22, widths=(64, 128, 256), dropout=0.0, num_classes=10, stem=16):
        super().__init__()
        assert (depth - 4) % 6 == 0, 'depth must be 6n+4'
        n = (depth - 4) // 6
        self.stem = nn.Conv2d(3, stem, 3, 1, 1, bias=False)
        layers, cin = [], stem
        for i, w in enumerate(widths):
            for j in range(n):
                layers.append(PreActBlock(cin, w, 2 if (i > 0 and j == 0) else 1, dropout)); cin = w
        self.blocks = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear): nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.blocks(self.stem(x))
        x = F.relu(self.bn(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

def build_student(): return PreActWRN(CFG['S_DEPTH'], CFG['S_WIDTHS'], CFG['S_DROPOUT'])
def build_teacher(): return PreActWRN(CFG['T_DEPTH'], CFG['T_WIDTHS'], CFG['T_DROPOUT'])

model = build_teacher() if CFG['MODE'] == 'teacher' else build_student()
n_params = count_params(model)
print(f"{CFG['MODE']} trainable params: {n_params:,}")
if CFG['MODE'] != 'teacher':
    assert n_params <= CFG['PARAM_LIMIT'], f'over budget: {n_params:,} > {CFG["PARAM_LIMIT"]:,}'
model = model.to(DEVICE).to(memory_format=torch.channels_last)

kd trainable params: 4,806,954


## Training utilities: mixing, SAM, EMA, KD

In [ ]:
def soft_targets(y, eps):
    t = torch.full((y.size(0), 10), eps / 10, device=y.device)
    return t.scatter_(1, y[:, None], 1 - eps + eps / 10)

def mix_batch(x, t):
    # returns mixed images and mixed soft targets (CutMix or MixUp)
    perm = torch.randperm(x.size(0), device=x.device)
    if random.random() < 0.5:
        lam = np.random.beta(CFG['CUTMIX_ALPHA'], CFG['CUTMIX_ALPHA'])
        H, W = x.shape[2:]
        rh, rw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy - rh // 2, 0), min(cy + rh // 2, H)
        x1, x2 = max(cx - rw // 2, 0), min(cx + rw // 2, W)
        x = x.clone(); x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    else:
        lam = np.random.beta(CFG['MIXUP_ALPHA'], CFG['MIXUP_ALPHA'])
        x = lam * x + (1 - lam) * x[perm]
    return x, lam * t + (1 - lam) * t[perm]

def soft_ce(logits, t): return torch.sum(-t * F.log_softmax(logits, 1), 1).mean()

def kd_loss(s_logits, t_logits, T):
    return F.kl_div(F.log_softmax(s_logits / T, 1), F.softmax(t_logits / T, 1), reduction='batchmean') * T * T

teacher = None
if CFG['MODE'] == 'kd':
    teacher = build_teacher()
    ck = torch.load(CFG['TEACHER_CKPT'], map_location='cpu', weights_only=False)
    teacher.load_state_dict(ck['ema'] if 'ema' in ck else ck['model'])
    teacher = teacher.to(DEVICE).to(memory_format=torch.channels_last).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    print('teacher loaded:', count_params(teacher), 'params (not part of final model)')

decay_params = [p for n, p in model.named_parameters() if p.ndim > 1]
no_decay     = [p for n, p in model.named_parameters() if p.ndim <= 1]   # BN and biases
opt = torch.optim.SGD([{'params': decay_params, 'weight_decay': CFG['WD']},
                       {'params': no_decay, 'weight_decay': 0.0}],
                      lr=CFG['LR'], momentum=CFG['MOMENTUM'], nesterov=True)
steps_per_epoch = len(train_dl); total = CFG['EPOCHS'] * steps_per_epoch; warm = CFG['WARMUP_EPOCHS'] * steps_per_epoch
def lr_lambda(s):
    if s < warm: return (s + 1) / warm
    return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
use_amp = CFG['AMP'] and DEVICE == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(CFG['EMA_DECAY']), use_buffers=True)

teacher loaded: 0 params (not part of final model)


In [ ]:
def train_step(x, y, mixing):
    t = soft_targets(y, CFG['LABEL_SMOOTH'])
    if mixing and random.random() < CFG['MIX_PROB']:
        x, t = mix_batch(x, t)
    t_out = None
    if teacher is not None:
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            t_out = teacher(x).float()

    def loss_fn():
        with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            out = model(x)
        out = out.float()
        l = soft_ce(out, t)
        if t_out is not None:
            l = CFG['KD_ALPHA'] * kd_loss(out, t_out, CFG['KD_T']) + (1 - CFG['KD_ALPHA']) * l
        return l, out

    loss, out = loss_fn()
    scaler.scale(loss).backward()
    if CFG['SAM']:
        scale = scaler.get_scale() if use_amp else 1.0
        params = [p for p in model.parameters() if p.grad is not None]
        gnorm = torch.norm(torch.stack([(p.grad.detach() / scale).norm(2) for p in params]), 2)
        if torch.isfinite(gnorm):
            eps = []
            with torch.no_grad():
                for p in params:
                    e = (p.grad / scale) * (CFG['SAM_RHO'] / (gnorm + 1e-12)); p.add_(e); eps.append(e)
            opt.zero_grad(set_to_none=True)
            loss2, _ = loss_fn()
            scaler.scale(loss2).backward()
            with torch.no_grad():
                for p, e in zip(params, eps): p.sub_(e)
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    sched.step(); ema.update_parameters(model)
    hard = t.argmax(1)
    return loss.item(), (out.argmax(1) == hard).float().mean().item()

@torch.no_grad()
def evaluate(net, dl):
    net.eval(); correct = n = 0
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            correct += (net(x).argmax(1) == y).sum().item(); n += y.numel()
    return correct / n

## Train

In [ ]:
ckpt_last = os.path.join(RUN_DIR, 'last.pt'); ckpt_best = os.path.join(RUN_DIR, 'best.pt')
start_epoch, best, log = 0, -1.0, []
if CFG['RESUME'] and os.path.exists(ckpt_last):
    ck = torch.load(ckpt_last, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model']); ema.module.load_state_dict(ck['ema'])
    ema.n_averaged.fill_(ck.get('n_averaged', 1))
    opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    start_epoch, best, log = ck['epoch'] + 1, ck['best'], ck['log']
    print('resumed at epoch', start_epoch)

mix_off_epoch = int(CFG['EPOCHS'] * (1 - CFG['MIX_OFF_LAST']))
for epoch in range(start_epoch, CFG['EPOCHS']):
    model.train(); t0 = time.time(); tl = ta = 0.0
    for x, y in train_dl:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE, non_blocking=True)
        l, a = train_step(x, y, mixing=epoch < mix_off_epoch); tl += l; ta += a
    row = dict(epoch=epoch, loss=tl / steps_per_epoch, train_acc=ta / steps_per_epoch,
               lr=sched.get_last_lr()[0], sec=time.time() - t0)
    last_epoch = epoch == CFG['EPOCHS'] - 1
    if val_dl is not None and ((epoch + 1) % CFG['EVAL_EVERY'] == 0 or last_epoch):
        row['val_acc'] = evaluate(model, val_dl); row['val_acc_ema'] = evaluate(ema.module, val_dl)
        if row['val_acc_ema'] > best:
            best = row['val_acc_ema']
            torch.save({'model': model.state_dict(), 'ema': ema.module.state_dict(), 'cfg': CFG, 'epoch': epoch}, ckpt_best)
    log.append(row); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    torch.save({'model': model.state_dict(), 'ema': ema.module.state_dict(), 'n_averaged': ema.n_averaged,
                'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                'epoch': epoch, 'best': best, 'log': log, 'cfg': CFG}, ckpt_last)
    pd.DataFrame(log).to_csv(os.path.join(RUN_DIR, 'log.csv'), index=False)

# final weights for prediction / as a KD teacher: EMA weights at the last epoch
final = os.path.join(RUN_DIR, 'final_ema.pt')
torch.save({'ema': ema.module.state_dict(), 'cfg': CFG}, final)
print('best val (EMA):', best, '| saved', final)

{'epoch': 0, 'loss': 1.512, 'train_acc': 0.2222, 'lr': 0.0201, 'sec': 13.0467}
{'epoch': 1, 'loss': 1.288, 'train_acc': 0.3783, 'lr': 0.0401, 'sec': 9.2642}
{'epoch': 2, 'loss': 1.159, 'train_acc': 0.4466, 'lr': 0.0601, 'sec': 9.3499}
{'epoch': 3, 'loss': 1.0624, 'train_acc': 0.4969, 'lr': 0.0801, 'sec': 9.0656}
{'epoch': 4, 'loss': 0.9921, 'train_acc': 0.5286, 'lr': 0.1, 'sec': 9.5404, 'val_acc': 0.708, 'val_acc_ema': 0.5626}
{'epoch': 5, 'loss': 0.9281, 'train_acc': 0.5634, 'lr': 0.1, 'sec': 9.3044}
{'epoch': 6, 'loss': 0.871, 'train_acc': 0.5904, 'lr': 0.1, 'sec': 9.4006}
{'epoch': 7, 'loss': 0.8317, 'train_acc': 0.6057, 'lr': 0.0999, 'sec': 9.2833}
{'epoch': 8, 'loss': 0.8102, 'train_acc': 0.611, 'lr': 0.0999, 'sec': 9.0423}
{'epoch': 9, 'loss': 0.7768, 'train_acc': 0.6273, 'lr': 0.0998, 'sec': 9.5474, 'val_acc': 0.7652, 'val_acc_ema': 0.8312}
{'epoch': 10, 'loss': 0.7519, 'train_acc': 0.6408, 'lr': 0.0998, 'sec': 9.0807}
{'epoch': 11, 'loss': 0.7421, 'train_acc': 0.6407, 'lr': 0.0

## Predict with TTA and write submission.csv

In [ ]:
@torch.no_grad()
def predict_tta(net, X_uint8, shifts=(0, 2)):
    # TTA views: identity, horizontal flip, and +-shift translations (with reflect padding) of both
    net.eval()
    x = torch.from_numpy(X_uint8).permute(0, 3, 1, 2).float().div(255)
    x = (x - torch.tensor(MEAN, dtype=torch.float32)[:, None, None]) / torch.tensor(STD, dtype=torch.float32)[:, None, None]
    probs = torch.zeros(len(x), 10)
    views = 0
    for i in range(0, len(x), 500):
        xb = x[i:i + 500].to(DEVICE)
        acc = torch.zeros(len(xb), 10, device=DEVICE)
        cand = [xb, xb.flip(3)]
        for s in shifts:
            if s == 0: continue
            pad = F.pad(xb, (s, s, s, s), mode='reflect')
            for dy, dx in [(0, s), (2 * s, s), (s, 0), (s, 2 * s)]:   # up, down, left, right
                v = pad[:, :, dy:dy + 32, dx:dx + 32]
                cand += [v, v.flip(3)]
        for v in cand:
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                acc += F.softmax(net(v.contiguous(memory_format=torch.channels_last)).float(), 1)
        probs[i:i + 500] = (acc / len(cand)).cpu(); views = len(cand)
    return probs.numpy(), views

net = build_teacher() if CFG['MODE'] == 'teacher' else build_student()
net.load_state_dict(torch.load(final, map_location='cpu', weights_only=False)['ema']); net = net.to(DEVICE).to(memory_format=torch.channels_last)

tta_shifts = (0, CFG['TTA_SHIFT']) if CFG['TTA_SHIFT'] else (0,)
if len(val_idx):
    res = {}
    for sh in [(0,), (0, 1), (0, 2)]:
        p, v = predict_tta(net, X_all[val_idx], sh)
        res[sh] = (p.argmax(1) == y_all[val_idx]).mean()
        print(f'val acc TTA shifts={sh} ({v} views): {res[sh]:.4f}')
    tta_shifts = max(res, key=res.get)
print('using TTA shifts', tta_shifts)
test_probs, _ = predict_tta(net, X_test, tta_shifts)
np.save(os.path.join(RUN_DIR, 'test_probs.npy'), test_probs)
pred = pd.DataFrame({'id': test_df['id'].values, 'label': test_probs.argmax(1)})
sub = sample_sub[['id']].merge(pred, on='id', how='left')
assert sub['label'].notna().all() and len(sub) == len(sample_sub)
sub['label'] = sub['label'].astype(int)
sub.to_csv(os.path.join(CFG['OUT_DIR'], 'submission.csv'), index=False)
sub.to_csv(os.path.join(RUN_DIR, 'submission.csv'), index=False)
print(sub.head(), '\nprediction class counts:', np.bincount(sub['label'], minlength=10))

val acc TTA shifts=(0,) (2 views): 0.9710
val acc TTA shifts=(0, 1) (10 views): 0.9700
val acc TTA shifts=(0, 2) (10 views): 0.9716
using TTA shifts (0, 2)
   id  label
0   0      6
1   1      3
2   2      3
3   3      9
4   4      3 
prediction class counts: [172 217 207 205 195 208 185 208 203 200]


## Notes
* **Parameter budget**: the assert above stops any student over 5,000,000 trainable params. An ensemble of
  several models counts as one final model, so the sum of all members must stay within 5M.
* **KD hygiene**: when validating a KD student, the teacher must be trained with `FULL_TRAIN=False` and the same
  `SEED` (same split), otherwise validation images leak through the teacher. For the final submission train both with `FULL_TRAIN=True`.
* **Public LB** is about 1,000 images (one image = 0.1%). Pick finals by local validation, not LB jumps.